# Day 2 — Knowledge Base, RAG & Property Intelligence
Production-Grade AI Voice Agent for Real Estate (UrduLish)

Tasks:
1. Knowledge Base (SQLite)
2. RAG Pipeline (ChromaDB)
3. Structured vs Semantic Retrieval
4. Recommendation Engine
5. Hallucination Evaluation

In [2]:
!pip install pandas chromadb sentence-transformers

In [3]:
import os, re, sqlite3
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_columns", None)
print("OK")

OK


## Task 1 — Knowledge Base (SQLite)

Categories: properties, prices, locations, amenities, payment_plans, developers, FAQs


In [4]:
CSV_FILES = [
    "lahore_properties.csv",
    "karachi_properties.csv",
    "islamabad_properties.csv",
    "rawalpindi_properties.csv",
]

dfs = []
for f in CSV_FILES:
    if os.path.exists(f):
        df = pd.read_csv(f)
        df["source_file"] = f
        dfs.append(df)
        print(f"Loaded {f}: {len(df)} rows")

props = pd.concat(dfs, ignore_index=True)
props = props.drop_duplicates(subset=["url"]).reset_index(drop=True)
print(f"\nTotal unique properties: {len(props)}")
props.head(3)

Loaded lahore_properties.csv: 125 rows
Loaded karachi_properties.csv: 125 rows
Loaded islamabad_properties.csv: 125 rows
Loaded rawalpindi_properties.csv: 125 rows

Total unique properties: 500


,city,title,price,location,beds,baths,area,added,url,source_file
0,Lahore,2-Kanal Ultra-Luxury Designer Fully Furnished ...,PKR 53.5 Crore,"DHA Phase 6, DHA Defence",6.0,7.0,2 Kanal,Added: 1 hour ago,https://www.zameen.com/Property/dha_defence_dh...,lahore_properties.csv
1,Lahore,1 Kanal Brand New Architectural Masterpiece Sp...,PKR 8.72 Crore,"DHA Phase 7 - Block Y, DHA Phase 7",5.0,6.0,1 Kanal,Added: 2 hours ago,https://www.zameen.com/Property/dha_phase_7_dh...,lahore_properties.csv
2,Lahore,1 KANAL HOUSE + 1 KANAL LAWN CORNER BRAND NEW ...,PKR 16.95 Crore,"DHA Phase 6, DHA Defence",6.0,7.0,2 Kanal,Added: 2 hours ago,https://www.zameen.com/Property/dha_defence_dh...,lahore_properties.csv


In [5]:
def parse_price(p):
    if pd.isna(p): return None
    s = str(p).lower().replace(",", "").strip()
    m = re.search(r"([\d.]+)\s*(crore|lakh|thousand|hundred)?", s)
    if not m: return None
    v = float(m.group(1))
    u = m.group(2)
    mult = {"crore":10_000_000, "lakh":100_000,
            "thousand":1_000, "hundred":100}.get(u, 1)
    return int(v * mult)

props["price_pkr"] = props["price"].apply(parse_price)
props[["title","price","price_pkr"]].head()

,title,price,price_pkr
0,2-Kanal Ultra-Luxury Designer Fully Furnished ...,PKR 53.5 Crore,535000000
1,1 Kanal Brand New Architectural Masterpiece Sp...,PKR 8.72 Crore,87200000
2,1 KANAL HOUSE + 1 KANAL LAWN CORNER BRAND NEW ...,PKR 16.95 Crore,169500000
3,5 Marla Brand New Luxury Ultra Modern Style Su...,PKR 3.85 Crore,38500000
4,16 Kanal Farm House Hudiara Barki Road FOR SALE,PKR 15 Crore,150000000


In [6]:
def parse_area(a):
    if pd.isna(a): return (None, None)
    s = str(a).lower().strip()
    m = re.search(r"([\d.,]+)\s*(marla|kanal|sq\.?\s*yd|sq\.?\s*ft)", s)
    if not m: return (None, None)
    try: v = float(m.group(1).replace(",", ""))
    except: return (None, None)
    unit = m.group(2).replace(".", "").replace(" ", "")
    return (v, unit)

props[["size","area_unit"]] = props["area"].apply(lambda x: pd.Series(parse_area(x)))
props[["title","area","size","area_unit"]].head()

,title,area,size,area_unit
0,2-Kanal Ultra-Luxury Designer Fully Furnished ...,2 Kanal,2.0,kanal
1,1 Kanal Brand New Architectural Masterpiece Sp...,1 Kanal,1.0,kanal
2,1 KANAL HOUSE + 1 KANAL LAWN CORNER BRAND NEW ...,2 Kanal,2.0,kanal
3,5 Marla Brand New Luxury Ultra Modern Style Su...,5 Marla,5.0,marla
4,16 Kanal Farm House Hudiara Barki Road FOR SALE,16 Kanal,16.0,kanal


In [7]:
DB = "knowledge_base.db"
if os.path.exists(DB): os.remove(DB)
conn = sqlite3.connect(DB)
cur = conn.cursor()

cur.executescript("""
CREATE TABLE properties (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    city TEXT, title TEXT, location TEXT, area TEXT,
    size REAL, area_unit TEXT, beds INTEGER, baths INTEGER,
    price TEXT, price_pkr INTEGER,
    purpose TEXT DEFAULT 'sale', status TEXT DEFAULT 'available',
    url TEXT UNIQUE, source_file TEXT
);
CREATE TABLE locations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    city TEXT, area TEXT, UNIQUE(city, area)
);
CREATE TABLE developers (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE, city TEXT, amenities TEXT, payment_plan TEXT
);
CREATE TABLE payment_plans (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT, plan_text TEXT
);
CREATE TABLE faqs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    category TEXT, question TEXT, answer TEXT
);
CREATE TABLE schools  (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT, city TEXT, area TEXT);
CREATE TABLE hospitals(id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT, city TEXT, area TEXT);
""")
conn.commit()
print("Tables created.")

Tables created.


In [8]:
rows = []
for _, r in props.iterrows():
    beds = int(r["beds"]) if pd.notna(r.get("beds")) and str(r["beds"]).strip() else None
    baths = int(r["baths"]) if pd.notna(r.get("baths")) and str(r["baths"]).strip() else None
    rows.append((r.get("city"), r.get("title"), r.get("location"), r.get("area"),
                 r.get("size"), r.get("area_unit"), beds, baths,
                 r.get("price"), r.get("price_pkr"),
                 "sale", "available", r.get("url"), r.get("source_file")))

cur.executemany("""
INSERT OR IGNORE INTO properties
(city,title,location,area,size,area_unit,beds,baths,
 price,price_pkr,purpose,status,url,source_file)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
""", rows)
conn.commit()
print(f"Inserted {cur.rowcount} properties.")

Inserted 500 properties.


In [9]:
loc_rows = props[["city","location"]].dropna().drop_duplicates().apply(tuple, axis=1).tolist()
cur.executemany("INSERT OR IGNORE INTO locations (city,area) VALUES (?,?)", loc_rows)
conn.commit()
print(f"Inserted {len(loc_rows)} locations.")

Inserted 258 locations.


In [10]:
if os.path.exists("real_estate_faqs.csv"):
    faq_df = pd.read_csv("real_estate_faqs.csv")
    faq_rows = faq_df[["category","question","answer"]].apply(tuple, axis=1).tolist()
    cur.executemany("INSERT INTO faqs (category,question,answer) VALUES (?,?,?)", faq_rows)
    conn.commit()
    print(f"Inserted {len(faq_rows)} FAQs.")
else:
    faq_df = pd.DataFrame()
    print("FAQ file not found.")

Inserted 22 FAQs.


In [11]:
if os.path.exists("zameen_projects_clean.csv"):
    proj = pd.read_csv("zameen_projects_clean.csv")
    dev_rows = proj[["developer","city","amenities","payment_plan_text"]].dropna(how="all")
    dev_rows = dev_rows.drop_duplicates(subset=["developer"])
    dev_rows = dev_rows[dev_rows["developer"].astype(str).str.strip() != ""]
    cur.executemany(
        "INSERT OR IGNORE INTO developers (name,city,amenities,payment_plan) VALUES (?,?,?,?)",
        dev_rows.apply(tuple, axis=1).tolist()
    )

    plan_rows = proj[["title","payment_plan_text"]].dropna()
    plan_rows = plan_rows[plan_rows["payment_plan_text"].astype(str).str.strip() != ""]
    cur.executemany(
        "INSERT INTO payment_plans (title,plan_text) VALUES (?,?)",
        plan_rows.apply(tuple, axis=1).tolist()
    )
    conn.commit()
    print(f"Developers: {len(dev_rows)} | Payment plans: {len(plan_rows)}")
else:
    print("Projects CSV not found.")

Developers: 753 | Payment plans: 750


In [12]:
print("--- 5 cheapest in Lahore ---")
display(pd.read_sql_query("""
SELECT title, price, price_pkr, beds, size, area_unit
FROM properties WHERE city='Lahore' AND price_pkr IS NOT NULL
ORDER BY price_pkr ASC LIMIT 5
""", conn))

print("\n--- Count per city ---")
display(pd.read_sql_query("SELECT city, COUNT(*) AS n FROM properties GROUP BY city", conn))

--- 5 cheapest in Lahore ---


,title,price,price_pkr,beds,size,area_unit
0,ONE BED LUXURIOUS FURNISHED APARTMENT FOR SALE...,PKR 45 Lakh,4500000,1.0,2.2,marla
1,A Beautiful Dream House Hot Location Brand New...,PKR 57 Lakh,5700000,3.0,3.0,marla
2,Good Location 1 Kanal Farm House In Cantt Of L...,PKR 88 Lakh,8800000,NaN,1.0,kanal
3,A Beautiful Dream House Hot Location,PKR 97 Lakh,9700000,3.0,3.0,marla
4,A Beautiful Dream House Hot Location Prime Loc...,PKR 1.05 Crore,10500000,7.0,5.0,marla



--- Count per city ---


,city,n
0,Islamabad,125
1,Karachi,125
2,Lahore,125
3,Rawalpindi,125


## Task 2 — RAG Pipeline (ChromaDB)

In [13]:
docs = []

# FAQs
for i, r in faq_df.iterrows():
    docs.append({
        "id": f"faq-{i}",
        "text": f"Q: {r['question']}\nA: {r['answer']}",
        "meta": {"type":"faq","category":str(r["category"])}
    })

# ALL properties (not just top 300)
for i, r in props.iterrows():
    docs.append({
        "id": f"prop-{i}",
        "text": f"{r['title']}. Located in {r['location']}, {r['city']}. "
                f"Size: {r['area']}, Beds: {r['beds']}, Baths: {r['baths']}, Price: {r['price']}.",
        "meta": {"type":"property","city":str(r["city"])}
    })

# Payment plans (from projects data, already in SQL)
if os.path.exists("zameen_projects_clean.csv"):
    proj = pd.read_csv("zameen_projects_clean.csv")
    for i, r in proj.dropna(subset=["payment_plan_text"]).iterrows():
        docs.append({
            "id": f"plan-{i}",
            "text": f"Payment plan for {r.get('title','this project')} "
                    f"({r.get('developer','')}, {r.get('city','')}): {r['payment_plan_text']}",
            "meta": {"type":"payment_plan","city":str(r.get("city",""))}
        })

# Developers (project-level amenities live here too)
    for i, r in proj.dropna(subset=["developer"]).drop_duplicates(subset=["developer"]).iterrows():
        docs.append({
            "id": f"dev-{i}",
            "text": f"{r['developer']} ({r.get('city','')}): "
                    f"Amenities include {r.get('amenities','not specified')}.",
            "meta": {"type":"developer","city":str(r.get("city",""))}
        })

print(f"Total docs: {len(docs)}")

Total docs: 2024


In [14]:
conn = sqlite3.connect(DB)
cur = conn.cursor()

In [15]:
cur.executescript("""
CREATE TABLE IF NOT EXISTS agents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT, city TEXT, phone TEXT, specialty TEXT
);
""")

# NOTE: scraper didn't capture per-listing agent names (zameen.com's listing
# cards don't expose agent names publicly on the search results page — only
# on the individual listing page). So this is a synthetic/demo agent roster,
# not real scraped data. Document this limitation in the report.
agents_data = [
    ("Ahmed Raza", "Lahore", "0300-1111111", "Residential"),
    ("Sana Khan", "Lahore", "0300-2222222", "Commercial"),
    ("Bilal Hussain", "Karachi", "0300-3333333", "Residential"),
    ("Ayesha Malik", "Karachi", "0300-4444444", "Investment"),
    ("Usman Tariq", "Islamabad", "0300-5555555", "Residential"),
    ("Hina Sheikh", "Islamabad", "0300-6666666", "Commercial"),
    ("Faisal Iqbal", "Rawalpindi", "0300-7777777", "Residential"),
]
cur.executemany("INSERT INTO agents (name, city, phone, specialty) VALUES (?,?,?,?)", agents_data)
conn.commit()

# assign agent to each property — round-robin within the same city
cur.execute("ALTER TABLE properties ADD COLUMN assigned_agent_id INTEGER")
props_by_city = pd.read_sql_query("SELECT id, city FROM properties", conn)
agents_by_city = pd.read_sql_query("SELECT id, city FROM agents", conn)

for city in props_by_city["city"].unique():
    city_agents = agents_by_city[agents_by_city["city"] == city]["id"].tolist()
    if not city_agents:
        continue  # no agent for this city — booking flow should escalate to "any available agent"
    city_props = props_by_city[props_by_city["city"] == city]["id"].tolist()
    for j, pid in enumerate(city_props):
        agent_id = city_agents[j % len(city_agents)]
        cur.execute("UPDATE properties SET assigned_agent_id=? WHERE id=?", (agent_id, pid))

conn.commit()
print("Agents table created and round-robin assigned.")

Agents table created and round-robin assigned.


In [16]:
cur.execute("ALTER TABLE properties ADD COLUMN amenities TEXT")

AMENITY_KEYWORDS = ["parking", "gym", "pool", "swimming pool", "security", "gated",
                    "lift", "elevator", "garden", "lawn", "basement", "servant quarter",
                    "store room", "furnished", "solar"]

def extract_amenities(title):
    title_l = str(title).lower()
    found = [kw for kw in AMENITY_KEYWORDS if kw in title_l]
    return ", ".join(found) if found else None

props["amenities"] = props["title"].apply(extract_amenities)

for _, r in props.dropna(subset=["amenities"]).iterrows():
    cur.execute("UPDATE properties SET amenities=? WHERE url=?", (r["amenities"], r["url"]))
conn.commit()

matched = props["amenities"].notna().sum()
print(f"Amenities extracted for {matched}/{len(props)} properties "
      f"(from listing title text — the scraper didn't collect a dedicated "
      f"amenities field, so this is keyword-based, not exhaustive).")

Amenities extracted for 66/500 properties (from listing title text — the scraper didn't collect a dedicated amenities field, so this is keyword-based, not exhaustive).


In [17]:
def chunk_text(text, size=400, overlap=50):
    words = text.split()
    step = max(1, size - overlap)
    return [" ".join(words[i:i+size]) for i in range(0, len(words), step) if words[i:i+size]]

In [18]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Dim:", embedder.get_sentence_embedding_dimension())

HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 4/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 5/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 2s [Retry 2/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 4s [Retry 3/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 8s [Retry 4/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 8s [Retry 5/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
HTTP Error 429

Dim: 384


C:\Users\cfiza\AppData\Local\Temp\ipykernel_8884\3420845476.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dim:", embedder.get_sentence_embedding_dimension())


In [19]:
def build_chroma(cs, ov, name):
    client = chromadb.Client()
    try: client.delete_collection(name)
    except: pass
    col = client.create_collection(name)
    texts, ids, metas = [], [], []
    for d in docs:
        for j, c in enumerate(chunk_text(d["text"], cs, ov)):
            texts.append(c); ids.append(f"{d['id']}-{j}"); metas.append(d["meta"])
    embs = embedder.encode(texts, show_progress_bar=False).tolist()
    col.add(documents=texts, embeddings=embs, ids=ids, metadatas=metas)
    return col, len(texts)

configs = [(200,30), (400,50), (800,100)]
cols = {}
for cs, ov in configs:
    c, n = build_chroma(cs, ov, f"kb_{cs}_{ov}")
    cols[(cs, ov)] = c
    print(f"chunk={cs}, overlap={ov} -> {n} chunks")

chunk=200, overlap=30 -> 2027 chunks
chunk=400, overlap=50 -> 2024 chunks
chunk=800, overlap=100 -> 2024 chunks


In [21]:
q = "What is required to rent a property?"
qe = embedder.encode([q]).tolist()
for (cs, ov), col in cols.items():
    print(f"\n=== chunk={cs}, overlap={ov} ===")
    for d in col.query(query_embeddings=qe, n_results=2)["documents"][0]:
        print(" -", d[:150], "...")


=== chunk=200, overlap=30 ===
 - Q: What is required to rent a property? A: Renting typically requires a security deposit (usually equal to 1-2 months' rent), advance rent payment, a  ...
 - Q: What documents are required to buy a property? A: You typically need a valid CNIC, proof of income or bank statement (if applying for financing), a ...

=== chunk=400, overlap=50 ===
 - Q: What is required to rent a property? A: Renting typically requires a security deposit (usually equal to 1-2 months' rent), advance rent payment, a  ...
 - Q: What documents are required to buy a property? A: You typically need a valid CNIC, proof of income or bank statement (if applying for financing), a ...

=== chunk=800, overlap=100 ===
 - Q: What is required to rent a property? A: Renting typically requires a security deposit (usually equal to 1-2 months' rent), advance rent payment, a  ...
 - Q: What documents are required to buy a property? A: You typically need a valid CNIC, proof of income or bank sta

In [22]:
BEST = (400, 50)
vectordb = cols[BEST]
print(f"Using chunk={BEST[0]}, overlap={BEST[1]}")

Using chunk=400, overlap=50


## Task 3 — Structured vs Semantic Retrieval
SQL → exact facts | Vector → open text
## Why SQL vs Vector — Justification

**SQL (structured) for:** Prices, plot sizes, bedroom/bathroom counts, agent names,
availability status.
- These are **exact, filterable, sortable facts** — "under 3 crore" or "4+ beds" needs
  a numeric comparison, which vector similarity search cannot do reliably. SQL gives
  100% precision here (no false matches).

**Vector (semantic) for:** FAQs, payment plan descriptions, developer/amenity blurbs.
- These are **free-text, paraphrase-prone questions** ("what's required to rent?" vs
  "rental requirements?" vs "kya chahiye rent pe lene ke liye?") — a keyword/SQL match
  would miss synonyms and code-switched Urdu phrasing. Embeddings capture *meaning*,
  not exact wording, which is what's needed here.

**Hybrid (both) — when it's used:** the `answer()` router in Task 3 checks for
property-search keywords first (price/beds/marla/budget) and routes to SQL; everything
else falls through to vector search. In a real call, a single customer turn often needs
both — e.g. "Show me DHA options under 3 crore" (SQL) followed immediately by "what's
their payment plan?" (vector, using the property found by the SQL step) — this two-step
pattern will be wired together properly in Day 5's LangGraph orchestration.

In [23]:
def is_structured(q):
    kw = ["price","kitne","how much","bedroom","bed ","bath","marla","kanal",
          "available","under","cheapest","lowest","sq","size","budget"]
    return any(k in q.lower() for k in kw)

def sql_search(city=None, max_price=None, min_beds=None, area_kw=None, limit=5):
    sql = "SELECT title, city, location, size, area_unit, beds, price, price_pkr FROM properties WHERE 1=1"
    p = []
    if city:       sql += " AND city=?";        p.append(city)
    if max_price:  sql += " AND price_pkr<=?";  p.append(max_price)
    if min_beds:   sql += " AND beds>=?";       p.append(min_beds)
    if area_kw:    sql += " AND (location LIKE ? OR title LIKE ?)"; p += [f"%{area_kw}%", f"%{area_kw}%"]
    sql += f" ORDER BY price_pkr ASC LIMIT {limit}"
    return pd.read_sql_query(sql, conn, params=p)

def vector_search(q, k=3):
    e = embedder.encode([q]).tolist()
    return vectordb.query(query_embeddings=e, n_results=k)["documents"][0]

def answer(q, **filters):
    if is_structured(q):
        print("[Router] STRUCTURED (SQL)")
        return sql_search(**filters)
    print("[Router] SEMANTIC (Vector)")
    return vector_search(q)

In [24]:
print("Q1:")
display(answer("5 marla house under 3 crore in Lahore",
               city="Lahore", max_price=30_000_000))

print("\nQ2:")
for d in answer("What is the security deposit for renting?"):
    print(" -", d[:180], "...")

Q1:
[Router] STRUCTURED (SQL)


,title,city,location,size,area_unit,beds,price,price_pkr
0,ONE BED LUXURIOUS FURNISHED APARTMENT FOR SALE...,Lahore,"Bahria Town - Sector E, Bahria Town",2.2,marla,1.0,PKR 45 Lakh,4500000
1,A Beautiful Dream House Hot Location Brand New...,Lahore,"Manawan, Lahore",3.0,marla,3.0,PKR 57 Lakh,5700000
2,Good Location 1 Kanal Farm House In Cantt Of L...,Lahore,"Lahore Organic Farms, Barki Road",1.0,kanal,NaN,PKR 88 Lakh,8800000
3,A Beautiful Dream House Hot Location,Lahore,"Ghous Garden - Phase 4, Ghous Garden",3.0,marla,3.0,PKR 97 Lakh,9700000
4,A Beautiful Dream House Hot Location Prime Loc...,Lahore,"Fahad Town, GT Road",5.0,marla,7.0,PKR 1.05 Crore,10500000



Q2:
[Router] SEMANTIC (Vector)
 - Q: What is required to rent a property? A: Renting typically requires a security deposit (usually equal to 1-2 months' rent), advance rent payment, a copy of your CNIC, and signing ...
 - Q: Is the security deposit refundable? A: Yes, the security deposit is refundable at the end of the tenancy, provided there is no damage to the property beyond normal wear and tear ...
 - Payment plan for Box Park III: Your Business Deserves a Place That Works as Hard as You Do (nan, Rawalpindi): kspaces that give startups, consultancies, and established firms a pro ...


## Task 4 — Recommendation Engine

In [25]:
def is_structured(q):
    kw = ["price","kitne","how much","bedroom","bed ","bath","marla","kanal",
          "under","cheapest","lowest","sq","size","budget"]
    return any(k in q.lower() for k in kw)

def sql_search(city=None, max_price=None, min_beds=None, area_kw=None, limit=5):
    sql = "SELECT title, city, location, size, area_unit, beds, price, price_pkr FROM properties WHERE 1=1"
    p = []
    if city:       sql += " AND city=?";        p.append(city)
    if max_price:  sql += " AND price_pkr<=?";  p.append(max_price)
    if min_beds:   sql += " AND beds>=?";       p.append(min_beds)
    if area_kw:    sql += " AND (location LIKE ? OR title LIKE ?)"; p += [f"%{area_kw}%", f"%{area_kw}%"]
    sql += f" ORDER BY price_pkr ASC LIMIT {limit}"
    return pd.read_sql_query(sql, conn, params=p)

def vector_search(q, k=3):
    e = embedder.encode([q]).tolist()
    return vectordb.query(query_embeddings=e, n_results=k)["documents"][0]

def answer(q, **filters):
    if is_structured(q):
        print("[Router] STRUCTURED (SQL)")
        return sql_search(**filters)
    print("[Router] SEMANTIC (Vector)")
    return vector_search(q)

In [26]:
def recommend(city=None, max_price=None, min_beds=None,
               area_kw=None, purpose=None, amenity_kw=None,
               investment_goal=None, limit=5):
    sql = "SELECT * FROM properties WHERE 1=1"
    p = []
    if city:       sql += " AND city=?"; p.append(city)
    if max_price:  sql += " AND price_pkr<=?"; p.append(max_price)
    if min_beds:   sql += " AND beds>=?"; p.append(min_beds)
    if area_kw:    sql += " AND (location LIKE ? OR title LIKE ?)"; p += [f"%{area_kw}%", f"%{area_kw}%"]
    if purpose:    sql += " AND purpose=?"; p.append(purpose)
    if amenity_kw: sql += " AND (title LIKE ? OR amenities LIKE ?)"; p += [f"%{amenity_kw}%", f"%{amenity_kw}%"]

    df = pd.read_sql_query(sql, conn, params=p)
    if df.empty:
        return df
    df = df.dropna(subset=["price_pkr"])
    if df.empty:
        return df

    def match_score(row):
        score, weights_total = 0.0, 0.0
        if max_price:
            weights_total += 1
            score += 1 - abs(max_price - row["price_pkr"]) / max_price
        if min_beds:
            weights_total += 1
            score += 1.0 if row["beds"] == min_beds else 0.7
        if area_kw:
            weights_total += 1
            hit = area_kw.lower() in str(row["location"]).lower() or area_kw.lower() in str(row["title"]).lower()
            score += 1.0 if hit else 0.3
        if amenity_kw:
            weights_total += 1
            score += 1.0 if amenity_kw.lower() in str(row.get("amenities","")).lower() else 0.0

        # NOTE: no rental-yield or historical-appreciation data was scraped,
        # so this is a documented heuristic, not a data-backed prediction:
        if investment_goal == "rental_income":
            weights_total += 1
            # smaller, cheaper units rent out faster / have more tenant demand
            score += 1.0 if row.get("size", 99) and row["size"] <= 5 else 0.5
        elif investment_goal == "resale_profit":
            weights_total += 1
            # larger plots in developing areas (DHA/Bahria) assumed higher appreciation
            hit = any(k in str(row["location"]).lower() for k in ["dha", "bahria", "phase"])
            score += 1.0 if hit else 0.4

        return score / weights_total if weights_total > 0 else (1 - row["price_pkr"] / df["price_pkr"].max())

    df["score"] = df.apply(match_score, axis=1)
    df = df.sort_values("score", ascending=False).head(limit)
    return df[["title","city","location","size","area_unit","beds","price","price_pkr","score"]]

## Task 5 — Hallucination Evaluation (20 questions)

In [27]:
questions = [
    "What is the security deposit for renting a property?",
    "How long does the registry take?",
    "Is bank financing available?",
    "What documents are needed to buy?",
    "How do I book a property?",
    "Is the security deposit refundable?",
    "Can rent be negotiated?",
    "What is the expected rental yield?",
    "Who handles maintenance after possession?",
    "What is the monthly maintenance charge?",
    "How do I reschedule a visit?",
    "Will an agent be available to show the property?",
    "Are there hidden charges?",
    "How can I verify a property is genuine?",
    "Is the project legally approved?",
    "Can a lawyer review the documents?",
    "What payment plans are available?",
    "What happens if I miss an installment?",
    "Is this a good time to invest?",
    "Can I visit before booking?",
]
print(len(questions), "questions ready")

20 questions ready


In [33]:
import google.generativeai as genai
genai.configure(api_key="")  # from aistudio.google.com
model = genai.GenerativeModel("gemini-3.5-flash-lite")

In [34]:
# ── Cell: Task 5 — Hallucination Evaluation (real LLM calls, rate-limit safe) ──
import time
import re
import pandas as pd

def real_answer(q, ctx, max_retries=3):
    context_text = "\n".join(c if isinstance(c, str) else str(c) for c in ctx[:3])
    prompt = f"""Answer the customer's question using ONLY the context below.
If the context doesn't contain the answer, say "I don't have that information."

Context:
{context_text}

Question: {q}
Answer (1-2 sentences):"""

    for attempt in range(max_retries):
        try:
            resp = model.generate_content(prompt)
            return resp.text.strip()
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                wait = 8 * (attempt + 1)   # 8s, 16s, 24s
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
    return "[Error: quota exceeded after retries]"


def check_grounding(answer, ctx):
    """Flag numbers in the answer that don't appear anywhere in the retrieved context."""
    context_text = " ".join(c if isinstance(c, str) else str(c) for c in ctx[:3])
    ans_nums = set(re.findall(r"\d+(?:\.\d+)?", answer))
    ctx_nums = set(re.findall(r"\d+(?:\.\d+)?", context_text))
    extra = ans_nums - ctx_nums - {"1","2","3","4","5","6","7","8","9","10"}
    return len(extra) == 0


eval_rows = []
for q in questions:
    route = "structured" if is_structured(q) else "vector"
    ctx = sql_search() if route == "structured" else vector_search(q)

    ans = real_answer(q, ctx)
    grounded = check_grounding(ans, ctx)

    eval_rows.append({
        "question": q,
        "answer": ans,
        "grounded": grounded,
        "route": route,
    })
    print(f"[{'OK' if grounded else 'FLAG'}] {q[:50]:50s} -> {ans[:60]}")

    time.sleep(5)   # stay under free-tier 15 requests/minute

df_eval = pd.DataFrame(eval_rows)
display(df_eval)

hallucination_rate = round((~df_eval["grounded"]).mean() * 100, 1)
grounding_rate = round(df_eval["grounded"].mean() * 100, 1)
print(f"\nGrounding Rate: {grounding_rate}%")
print(f"Hallucination Rate: {hallucination_rate}%")

[OK] What is the security deposit for renting a propert -> Renting a property typically requires a security deposit tha
[OK] How long does the registry take?                   -> The registry process usually takes 2 to 6 weeks after full p
[OK] Is bank financing available?                       -> Yes, bank financing is available through many banks in Pakis
[OK] What documents are needed to buy?                  -> To buy a property, you typically need a valid CNIC and, if a
[OK] How do I book a property?                          -> A property is booked by paying a token or booking amount (us
[OK] Is the security deposit refundable?                -> Yes, the security deposit is refundable at the end of the te
[OK] Can rent be negotiated?                            -> Rent is often somewhat negotiable depending on the landlord 
[OK] What is the expected rental yield?                 -> Rental yields generally range between 3-6% annually in most 
[OK] Who handles maintenance after posse

,question,answer,grounded,route
0,What is the security deposit for renting a pro...,Renting a property typically requires a securi...,True,vector
1,How long does the registry take?,The registry process usually takes 2 to 6 week...,True,vector
2,Is bank financing available?,"Yes, bank financing is available through many ...",True,vector
3,What documents are needed to buy?,"To buy a property, you typically need a valid ...",True,vector
4,How do I book a property?,A property is booked by paying a token or book...,True,vector
5,Is the security deposit refundable?,"Yes, the security deposit is refundable at the...",True,vector
6,Can rent be negotiated?,Rent is often somewhat negotiable depending on...,True,vector
7,What is the expected rental yield?,Rental yields generally range between 3-6% ann...,True,vector
8,Who handles maintenance after possession?,Maintenance responsibility depends on the soci...,True,vector
9,What is the monthly maintenance charge?,Maintenance charges vary by society and proper...,True,vector



Grounding Rate: 100.0%
Hallucination Rate: 0.0%


In [35]:
# ── Cell: Save (same as before) ──
df_eval.to_csv("day2_hallucination_report.csv", index=False)
print("Saved day2_hallucination_report.csv")
conn.close()
print("Day 2 COMPLETE.")

Saved day2_hallucination_report.csv
Day 2 COMPLETE.
